In [1]:
import os
import pandas as pd
import numpy as np
from openpyxl import load_workbook, Workbook
from datetime import datetime

In [2]:
def concat_files_html(folder_path, pattern, row):
    files = [f for f in os.listdir(folder_path) if pattern in f]
    df_combine = pd.DataFrame()

    for filename in files:
        file_path = os.path.join(folder_path, filename)
        print(f"Processing file: {file_path}")
        
        try:
            tables = pd.read_html(file_path, header=row)
            print(f"Found {len(tables)} tables in {filename}")
            
            # Duyệt từng bảng và nối vào
            for table in tables:
                table["Source"] = filename
                df_combine = pd.concat([df_combine, table], ignore_index=True)
            
            print(f"Processed file: {filename}")
        except Exception as e:
            print(f"Failed to process file {filename}: {e}")
    
    return df_combine


In [3]:
def concat_files_txt(folder_path, pattern,row_to_skip):
    files = [f for f in os.listdir(folder_path) if pattern in f]
    df_combine = pd.DataFrame()
    df_combine = df_combine.iloc[0:0]
    for filename in files:
        file_path = os.path.join(folder_path, filename)
        print(f"Processing file: {file_path}")
        
        try:
            df = pd.read_fwf(file_path, skiprows= row_to_skip)
            df["Source"] = filename
            df_combine = pd.concat([df_combine, df], ignore_index=True)
            print(f"Processed file: {filename}")
        except Exception as e:
            print(f"Failed to process file {filename}: {e}")
    
    return df_combine

# Source

In [4]:
folder_path = r"C:\Users\PMCTKLNGUYEN02\OneDrive - Techtronic Industries Co. Ltd\02. MC\ECO\File Run\Import"

# 124520
pattern = "ECO_Detail"
ECO_Detail = concat_files_html(folder_path,pattern,0)

# Revised item status
# df_all_ECO = pd.read_csv(r"C:\Users\PMCTKLNGUYEN02\OneDrive - Techtronic Industries Co. Ltd\02. MC\ECO\File Run\Import\All_ECO.tsv", sep="\t", encoding="utf-16")

# 10810

folder_path = r"C:\Users\PMCTKLNGUYEN02\OneDrive - Techtronic Industries Co. Ltd\02. MC\ECO\File Run\Import"

# Lặp qua các file trong thư mục
file_name = None
for f in os.listdir(folder_path):
    if "ECO_Approved_Status_List" in f and f.endswith(".xls"):
        file_name = os.path.join(folder_path, f)
        break  # Dừng lại sau khi tìm thấy file đầu tiên phù hợp

# Đọc file nếu tìm thấy
if file_name:
    Approved_Status = pd.read_fwf(file_name, header=2)
else:
    print("Không tìm thấy file phù hợp.")


Approved_Status[['Approved', 'Status', 'Eng MGR']] = Approved_Status['Approved      Status     Eng MGR'].str.split(expand=True, n=2)
Approved_Status = Approved_Status[['ECO No.', 'Requestor', 'Creation', 'Approved', 'Status', 'Eng MGR','QA MGR', 'PMC MGR',  'DCC MGR', 'ENG MGR', 'QA MGR','PMC MGR', 'COST MGR', 'DCC MGR.1', 'EFFECTIVE_D']]
print(Approved_Status.columns)
Approved_Status = Approved_Status[
    ~Approved_Status["ECO No."].astype(str).str.startswith("ECO") &
    (Approved_Status["ECO No."].astype(str).str.strip() != "") &
    (Approved_Status["ECO No."].astype(str).str.strip() != "-------")
]

# all item report
# pattern = "All_Items"
# df_org_all_items = concat_files_html(folder_path, pattern,1)

df_org_all_items = pd.read_excel(r"C:\Users\PMCTKLNGUYEN02\OneDrive - Techtronic Industries Co. Ltd\02. MC\ECO\File Run\Import\All_Item.xlsx", header = 3)
df_org_all_items['SOURCING_RULE'] = df_org_all_items['SOURCING_RULE'].str.extract(r'\b([A-Z]{3}-\d+-[AB])\b')
df_all_items = df_org_all_items[["ITEM_NO","ITEM_DESC", "Make/Buy","SOURCING_RULE","BUYER","Planner"]]
# input date (PMC received date)
input_date = pd.read_excel(r"C:\Users\PMCTKLNGUYEN02\OneDrive - Techtronic Industries Co. Ltd\02. MC\ECO\File Run\Import\input_date.xlsx", header = 0)


Processing file: C:\Users\PMCTKLNGUYEN02\OneDrive - Techtronic Industries Co. Ltd\02. MC\ECO\File Run\Import\MIS__SCP_ECO_Detail_report_for_170725. (1).xls
Found 1 tables in MIS__SCP_ECO_Detail_report_for_170725. (1).xls
Processed file: MIS__SCP_ECO_Detail_report_for_170725. (1).xls
Index(['ECO No.', 'Requestor', 'Creation', 'Approved', 'Status', 'Eng MGR',
       'QA MGR', 'PMC MGR', 'DCC MGR', 'ENG MGR', 'QA MGR', 'PMC MGR',
       'COST MGR', 'DCC MGR.1', 'EFFECTIVE_D'],
      dtype='object')


In [5]:
# ECO summary rule
data_rule = [
    {'ECO_summary': 'Add', 'BOM_diff_Sum': True, 'ECO_change_summary': 'Add OR'},
    {'ECO_summary': 'Add', 'BOM_diff_Sum': False, 'ECO_change_summary': 'Add item'},
    {'ECO_summary': 'Add|Change', 'BOM_diff_Sum': False, 'ECO_change_summary': 'Change Item'},
    {'ECO_summary': 'Add|Change', 'BOM_diff_Sum': True, 'ECO_change_summary': 'Change EI/OR'},
    {'ECO_summary': 'Add|Change REV', 'BOM_diff_Sum': True, 'ECO_change_summary': 'Change Item + REV'},
    {'ECO_summary': 'Add|Change REV', 'BOM_diff_Sum': False, 'ECO_change_summary': 'Change Item + REV'},
    {'ECO_summary': 'Add|Change REV|Delete', 'BOM_diff_Sum': True, 'ECO_change_summary': 'Change Item + REV'},
    {'ECO_summary': 'Add|Change REV|Delete', 'BOM_diff_Sum': False, 'ECO_change_summary': 'Change Item + REV'},
    {'ECO_summary': 'Add|Change|Change REV', 'BOM_diff_Sum': True, 'ECO_change_summary': 'Change Item + REV'},
    {'ECO_summary': 'Add|Change|Change REV', 'BOM_diff_Sum': False, 'ECO_change_summary': 'Change Item + REV'},
    {'ECO_summary': 'Add|Change|Change REV|Delete', 'BOM_diff_Sum': True, 'ECO_change_summary': 'Change Item + REV'},
    {'ECO_summary': 'Add|Change|Change REV|Delete', 'BOM_diff_Sum': False, 'ECO_change_summary': 'Change Item + REV'},
    {'ECO_summary': 'Add|Change|Delete', 'BOM_diff_Sum': True, 'ECO_change_summary': 'Change item'},
    {'ECO_summary': 'Add|Change|Delete', 'BOM_diff_Sum': False, 'ECO_change_summary': 'Change item'},
    {'ECO_summary': 'Add|Delete', 'BOM_diff_Sum': True, 'ECO_change_summary': 'Change item'},
    {'ECO_summary': 'Add|Delete', 'BOM_diff_Sum': False, 'ECO_change_summary': 'Change item'},
    {'ECO_summary': 'Change', 'BOM_diff_Sum': False, 'ECO_change_summary': 'Change usage'},
    {'ECO_summary': 'Change', 'BOM_diff_Sum': True, 'ECO_change_summary': 'Update EI'},
    {'ECO_summary': 'Change REV', 'BOM_diff_Sum': True, 'ECO_change_summary': 'Change REV'},
    {'ECO_summary': 'Change REV|Delete', 'BOM_diff_Sum': False, 'ECO_change_summary': 'Change Item + REV'},
    {'ECO_summary': 'Change REV|Delete', 'BOM_diff_Sum': True, 'ECO_change_summary': 'Change Item + REV'},
    {'ECO_summary': 'Change|Change REV', 'BOM_diff_Sum': True, 'ECO_change_summary': 'Change Item + REV'},
    {'ECO_summary': 'Change|Change REV', 'BOM_diff_Sum': False, 'ECO_change_summary': 'Change Item + REV'},
    {'ECO_summary': 'Change|Change REV|Delete', 'BOM_diff_Sum': True, 'ECO_change_summary': 'Change Item + REV'},
    {'ECO_summary': 'Change|Change REV|Delete', 'BOM_diff_Sum': False, 'ECO_change_summary': 'Change Item + REV'},
    {'ECO_summary': 'Change|Delete', 'BOM_diff_Sum': True, 'ECO_change_summary': 'Change item'},
    {'ECO_summary': 'Change|Delete', 'BOM_diff_Sum': False, 'ECO_change_summary': 'Change item'},
    {'ECO_summary': 'Delete', 'BOM_diff_Sum': False, 'ECO_change_summary': 'Delete item'},
    {'ECO_summary': 'Delete', 'BOM_diff_Sum': True, 'ECO_change_summary': 'Delete item'}
]

df_rule = pd.DataFrame(data_rule)


# Calculator

## ECO still open

In [6]:

dropcol = ['C_IQC', 'C_ONHAND', 'C_SUPPLY','Com Item SER_NO', 'Com item LIMIT_QTY',
       'WEEK01_DEMAND', 'WEEK02_DEMAND', 'WEEK03_DEMAND', 'WEEK04_DEMAND',
       'C_DEMAND_DATE', 'C_DEMAND', 'C_PO_DATE', 'C_ASL_VENDOR_CODE',
       'C_ASL_VENDOR_NAME',  'R_IQC', 'R_ONHAND', 'R_SUPPLY',
       'R_DEMAND', 'R_DEMAND_DATE', 'R_ASL_VENDOR_CODE', 'R_ASL_VENDOR_NAME',
       'R_PO_DATE', 'R_LAST_PO_VENDOR_CODE', 'R_LAST_PO_VENDOR_NAME',
       'C_LAST_PO_VENDOR_CODE', 'C_LAST_PO_VENDOR_NAME','REVISED Item SER_NO','REVISED item LIMIT_QTY','USE_UP_ITEM','PMC_NEW_ITEM','CUT_OFF_DATE','CUT_OFF_ONHAND'
]

df_ECO = ECO_Detail.drop(columns=dropcol)
# df_ECO_open = df_ECO.copy()
df_ECO_open = df_ECO[df_ECO["ECO_STATUS_TYPE"] != "Implemented"]
df_ECO_open['ACTION_TYPE'] = df_ECO_open['ACTION_TYPE'].fillna('Change REV')
df_ECO_open['BOM_diff']=(df_ECO_open['NEW_QTY']-df_ECO_open['OLD_QTY'])

df_ECO_open['BOM_diff_Sum'] = (
    df_ECO_open.groupby(['REVISED_ITEM', 'ECO_NO'])['BOM_diff']
    .transform('sum')
)

df_ECO_open = df_ECO_open.assign(
    ECO_summary = df_ECO_open['ECO_NO'].map(
        df_ECO_open.drop_duplicates()
        .groupby('ECO_NO')['ACTION_TYPE']
        .apply(lambda x: '|'.join(sorted(x.unique())))
    )
)


# Trong df_rule, đổi tên cột cho khớp với df_ECO_open
df_ECO_open['BOM_diff_Sum_flag'] = df_ECO_open['BOM_diff_Sum'] == 0

df_rule.rename(columns={'BOM_diff_Sum': 'BOM_diff_Sum_flag'}, inplace=True)
df_ECO_open = df_ECO_open.merge(df_rule, how='left', on=['ECO_summary', 'BOM_diff_Sum_flag'])
dropcol = ['Source','BOM_diff','BOM_diff_Sum','BOM_diff_Sum_flag']
df_ECO_open.drop(columns=dropcol, inplace=True)

df_ECO_open = df_ECO_open.merge(input_date, on='ECO_NO', how='left')
today_str = datetime.today().strftime('%d-%b-%y')
df_ECO_open['Input Date'].fillna(today_str, inplace=True)


# with pd.ExcelWriter(r"C:\Users\PMCTKLNGUYEN02\OneDrive - Techtronic Industries Co. Ltd\02. MC\ECO\File Run\Import\input_date.xlsx") as writer:
#     input_date_1.to_excel(writer, sheet_name='Sheet1', index=False)

# Đưa các cột 'Email' và 'Email name' lên đầu
cols = df_ECO_open.columns.tolist()
new_order = ['Input Date','ECO_summary', 'ECO_change_summary'] + [col for col in cols if col not in ['Input Date','ECO_summary', 'ECO_change_summary']]
df_ECO_open = df_ECO_open[new_order]



C:\Users\PMCTKLNGUYEN02\AppData\Local\Temp\ipykernel_14576\1601531676.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ECO_open['ACTION_TYPE'] = df_ECO_open['ACTION_TYPE'].fillna('Change REV')
C:\Users\PMCTKLNGUYEN02\AppData\Local\Temp\ipykernel_14576\1601531676.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ECO_open['BOM_diff']=(df_ECO_open['NEW_QTY']-df_ECO_open['OLD_QTY'])
C:\Users\PMCTKLNGUYEN02\AppData\Local\Temp\ipykernel_14576\1601531676.py:16: SettingWithCopyWarning: 
A value is tryi

In [7]:
input_date_1 = df_ECO_open[['ECO_NO','Input Date', 'ECO_APPROVAL_STATUS']]
input_date_1 = input_date_1[input_date_1['Input Date'] == today_str]
input_date_1 = input_date_1[input_date_1['ECO_APPROVAL_STATUS'] == 'Approval requested']
input_date_1.drop(columns='ECO_APPROVAL_STATUS', inplace=True)
input_date_1.drop_duplicates(inplace=True)
input_date =pd.concat([input_date, input_date_1],ignore_index=True)
input_date.to_excel(r"C:\Users\PMCTKLNGUYEN02\OneDrive - Techtronic Industries Co. Ltd\02. MC\ECO\File Run\Import\input_date.xlsx",index=False)

## Make part to compare BOM

In [8]:


# 1. Lấy cột COMPONENT_ITEM, loại trùng
Mark_part = df_ECO_open[["COMPONENT_ITEM","C_MAKE_BUY"]].copy().drop_duplicates()
Mark_part =  Mark_part[Mark_part["C_MAKE_BUY"]=="Make"]
# 2. Thêm cột BU
Mark_part["BU"] = "TC2"
Mark_part.drop(columns=['C_MAKE_BUY'], inplace= True)
# 3. Đảm bảo thứ tự cột đúng (nếu cần)
Mark_part = Mark_part[["BU", "COMPONENT_ITEM"]]

output_file = r"C:\Users\PMCTKLNGUYEN02\OneDrive - Techtronic Industries Co. Ltd\02. MC\ECO\File Run\Import\Mark_part.txt"
Mark_part.to_csv(output_file, sep='\t', index=False)


## Compared BOM

In [9]:
pattern = "Bom_Data"
BOM_make_part = concat_files_html(folder_path,pattern,0)

rename_dict ={'TOP_MODEL': 'COMPONENT_ITEM','COMPONENT':'BUY_PART' }
BOM_make_part.rename(columns=rename_dict, inplace=True)
dropcol = ['ORGANIZATION_CODE',
       'SOURCING_SUB_CATEGORY', 'DESCRIPTION', 'REVISION', 'BOM TYPE',
       'PLANNER CODE',  'unit_weight', 'weight_uom_code', 'MOQ',
       'Demand Qty', 'Demand last date', 'Supply Qty', 'IQC Qty', 'Onhand Qty',
       'LEADTIME', 'ITEM_COST', 'CURRENCY_CODE', 'SR_VENDOR_CODE',
       'SR_VENDOR_NAME', 'VENDOR_SITE_CODE', 'SER', 'SER Status', 'IRS',
       'Source']

BOM_make_part.drop(columns=dropcol, inplace=True)
BOM_make_part["COMPONENT_ITEM"] = BOM_make_part["COMPONENT_ITEM"] .astype(str).str.lstrip("'")
BOM_make_part["BUY_PART"] = BOM_make_part["BUY_PART"] .astype(str)
BOM_make_part["C_MAKE_BUY"]="Make"

Processing file: C:\Users\PMCTKLNGUYEN02\OneDrive - Techtronic Industries Co. Ltd\02. MC\ECO\File Run\Import\MIS__SCP_Bom_Data_For_PMC_Repo_170725..xls
Found 1 tables in MIS__SCP_Bom_Data_For_PMC_Repo_170725..xls
Processed file: MIS__SCP_Bom_Data_For_PMC_Repo_170725..xls


## Merged compared BOM with ECOs open

In [10]:
df_ECO_make = df_ECO_open.merge(BOM_make_part, on=["COMPONENT_ITEM","C_MAKE_BUY"], how = 'left')

keepcols=['ECO_NO', 
       'ECO_EFFECTIVE_TYPE', 'ECO_STATUS_TYPE','REVISED_ITEM', 'DESCRIPTION',
       'C_MAKE_BUY',  'COMPONENT_ITEM', 'OLD_QTY', 'NEW_QTY',  'BUY_PART',
       'BOM Qty','ACTION_TYPE','BUYER CODE']
df_ECO_make = df_ECO_make[keepcols]
df_ECO_make = df_ECO_make[df_ECO_make['C_MAKE_BUY'] == "Make"]
df_ECO_make['BOM Qty']=df_ECO_make['BOM Qty'].astype(float).round(6)
df_ECO_make['NEW_QTY']=df_ECO_make['NEW_QTY'].astype(float).round(6)
df_ECO_make['OLD_QTY']=df_ECO_make['OLD_QTY'].astype(float).round(6)
df_ECO_make['BOM_Change']=(df_ECO_make['NEW_QTY']-df_ECO_make['OLD_QTY'])*df_ECO_make['BOM Qty']
df_ECO_make = df_ECO_make.drop_duplicates()
# df_ECO_make['BUY_PART_BOM_Sum'] = df_ECO_make.groupby(['BUY_PART'])['BOM_Change'].transform('sum')
df_ECO_make=df_ECO_make[df_ECO_make['BOM_Change'] !=0]

df_ECO_make1=df_ECO_make.copy()
# Làm tròn tổng BOM change sau khi group
df_ECO_make1['BUY_PART_BOM_Sum'] = (
    df_ECO_make1.groupby(['BUY_PART', 'ECO_NO','REVISED_ITEM'])['BOM_Change']
    .transform('sum')
    .round(6)
)

# Lọc ra các dòng có tổng khác 0 thực sự
df_ECO_make1 = df_ECO_make1[df_ECO_make1['BUY_PART_BOM_Sum'] != 0]

## ECO tracking file

In [11]:
ECO_Tracking_ECOs = df_ECO_open.merge(df_ECO_make1,on=['ECO_NO', 'ECO_EFFECTIVE_TYPE', 'ECO_STATUS_TYPE', 'DESCRIPTION',
       'C_MAKE_BUY', 'COMPONENT_ITEM', 'OLD_QTY', 'NEW_QTY','ACTION_TYPE'], how = 'left')

In [12]:
ECO_Tracking_ECOs['MC']=ECO_Tracking_ECOs.apply(
    lambda row: row['R_BUYER'] if (row['ACTION_TYPE'] == 'Change REV') else row['BUYER CODE'] if (row['C_MAKE_BUY'] == 'Make') else row['C_BUYER'], 
    axis=1
)
ECO_Tracking_ECOs.drop(columns=['REVISED_ITEM_y','CURRENT_ITEM_REV','C_PLANER','R_PLANER','ECO_CREATION_DATE','BUYER CODE'], inplace=True)

print(ECO_Tracking_ECOs.columns)

ECO_Tracking_ECOs['BUY_PART'] = ECO_Tracking_ECOs.apply(
    lambda row: row['REVISED_ITEM_x'] if row['ACTION_TYPE'] == 'Change REV' else row['BUY_PART'],
    axis=1
)

ECO_Tracking_ECOs['ITEM_NO']= ECO_Tracking_ECOs.apply(
    lambda row: row['REVISED_ITEM_x'] if (row['ACTION_TYPE'] == 'Change REV') else row['BUY_PART'] if (row['C_MAKE_BUY'] == 'Make') else row['COMPONENT_ITEM'], 
    axis=1
)

ECO_Tracking_ECOs["ITEM_NO"] = ECO_Tracking_ECOs["ITEM_NO"].fillna("")

# Làm sạch và lọc luôn
# df_all_items["ITEM_NO"] = df_all_items["ITEM_NO"].astype(str).str.strip()

# df_all_items = df_all_items[df_all_items["ITEM_NO"].str.match(r'^\d{9}$')]
# ECO_Tracking_ECOs["ITEM_NO"] = ECO_Tracking_ECOs["ITEM_NO"].astype(str).str.strip()
# ECO_Tracking_ECOs = ECO_Tracking_ECOs[ECO_Tracking_ECOs["ITEM_NO"].str.match(r'^\d{9}$')]

ECO_Tracking_ECOs= ECO_Tracking_ECOs.merge(df_all_items, on='ITEM_NO', how='left')

ECO_by_MC = (
    ECO_Tracking_ECOs.groupby(['Input Date',  'ECO_change_summary', 'ECO_NO',
       'ECO Requestor', 'ECO_EFFECTIVE_TYPE', 'ECO_STATUS_TYPE',  
       'REVISED_ITEM_CUR_REV', 'COMPONENT_ITEM', 'OLD_QTY', 'NEW_QTY',
       'ACTION_TYPE', 'EFFECTIVITY_DATE', 'SCHEDULED_DATE', 'DESCRIPTION',
       'C_MAKE_BUY', 'R_MAKE_BUY',         'ECO_APPROVAL_STATUS',
       'ECO_IMPLETEMENT_DATE', 'BUY_PART', 'ITEM_DESC','BOM Qty','BOM_Change','SOURCING_RULE' ,'Planner',
       'MC'])['BUY_PART_BOM_Sum']
    .transform('count')
)

Index(['Input Date', 'ECO_summary', 'ECO_change_summary', 'ORG_CODE', 'ECO_NO',
       'ECO Requestor', 'ECO REASON', 'ECO_EFFECTIVE_TYPE', 'ECO_STATUS_TYPE',
       'WHEREUSE', 'REVISED_ITEM_x', 'REVISED_ITEM_CUR_REV', 'COMPONENT_ITEM',
       'OLD_QTY', 'NEW_QTY', 'ACTION_TYPE', 'EFFECTIVITY_DATE',
       'SCHEDULED_DATE', 'DESCRIPTION', 'C_BUYER', 'C_MAKE_BUY', 'R_MAKE_BUY',
       'R_BUYER', 'ECO_APPROVAL_STATUS', 'ECO_IMPLETEMENT_DATE', 'BUY_PART',
       'BOM Qty', 'BOM_Change', 'BUY_PART_BOM_Sum', 'MC'],
      dtype='object')


## ECO by MC and Buy part level

In [13]:
ECO_by_MC =    ECO_Tracking_ECOs [[
        'Input Date',   'ECO_NO','ECO_change_summary',
        'ECO_EFFECTIVE_TYPE', 'ECO_STATUS_TYPE',
        'REVISED_ITEM_CUR_REV', 'COMPONENT_ITEM', 'OLD_QTY', 'NEW_QTY',
        'ACTION_TYPE', 'EFFECTIVITY_DATE', 'SCHEDULED_DATE', 'DESCRIPTION',
        'C_MAKE_BUY', 'ECO_APPROVAL_STATUS', 'BUY_PART', 'BOM Qty','BOM_Change','ITEM_DESC','SOURCING_RULE','Planner',
        'MC'    ]].drop_duplicates()

ECO_by_MC = ECO_by_MC[ECO_by_MC['ECO_STATUS_TYPE'] !='In Progress']

ECO_by_MC['Index']=ECO_by_MC['ECO_NO'].astype(str) +  ECO_by_MC['MC'].astype(str)
ECO_by_MC_rejected = ECO_by_MC[ECO_by_MC['ECO_APPROVAL_STATUS'] == 'Rejected']
ECO_by_MC = ECO_by_MC[ECO_by_MC['ECO_APPROVAL_STATUS'] !='Rejected']



In [14]:
ECO_replying_tracking = ECO_by_MC[[
        'Input Date',   'ECO_NO','ECO_change_summary',
        'ECO_EFFECTIVE_TYPE', 'ECO_STATUS_TYPE', 'EFFECTIVITY_DATE', 'SCHEDULED_DATE',
        'ECO_APPROVAL_STATUS',         'MC'    ]].drop_duplicates()
Buyer_info = pd.read_excel(r"C:\Users\PMCTKLNGUYEN02\OneDrive - Techtronic Industries Co. Ltd\02. MC\ECO\File Run\Import\Buyer info.xlsx",header =0)
ECO_replying_tracking = ECO_replying_tracking.merge(Buyer_info, left_on='MC', right_on='Buyer', how= 'left')

ECO_replying_tracking['Index'] = (
    ECO_replying_tracking['ECO_NO'].fillna("").astype(str)
    + ECO_replying_tracking['Buyer'].fillna("").astype(str)
)

ECO_replying_tracking.drop(columns=['MC','Email name'], inplace= True)


# Output

In [15]:
output_file = r"C:\Users\PMCTKLNGUYEN02\OneDrive - Techtronic Industries Co. Ltd\02. MC\ECO\File Run\Import\Tracking File.xlsx"
    # Create an Excel writer for the current vendor
with pd.ExcelWriter(output_file) as writer:
    ECO_Detail.to_excel(writer, sheet_name='Org', index=False)
    df_ECO_open.to_excel(writer, sheet_name='Open_ECO', index=False)
    df_ECO_make1.to_excel(writer, sheet_name='BOM_Compared', index=False)
    ECO_Tracking_ECOs.to_excel(writer, sheet_name='ECO Tracking', index=False)
    ECO_by_MC.to_excel(writer, sheet_name='ECO Pending - Component level', index=False)
    ECO_replying_tracking.to_excel(writer, sheet_name='ECO Pending - MC level', index=False)
    ECO_by_MC_rejected.to_excel(writer, sheet_name='Rejected', index=False)
    
print(f"Data exported to '{output_file}'")

Data exported to 'C:\Users\PMCTKLNGUYEN02\OneDrive - Techtronic Industries Co. Ltd\02. MC\ECO\File Run\Import\Tracking File.xlsx'
